# Biohub — Cell Tracking During Development: baseline submission

Classical-CV baseline (no training data / pretrained model required):

1. **Detect**: per-timepoint 3D blob detection — anisotropic Gaussian smoothing,
   Otsu/relative thresholding, physically-spaced local maxima, marker-controlled
   watershed, intensity-weighted centroids.
2. **Link**: frame-to-frame 1-1 assignment via the Hungarian algorithm on
   physically-scaled centroid distance, with dummy rows/columns so cells can
   appear/disappear instead of being forced into a bad match.
3. **Divide**: recover extra parent→daughter edges for any daughter left
   unmatched by the 1-1 step, giving dividing cells two outgoing edges.
4. **Submit**: format nodes + edges into the required `submission.csv`.

Runs with no internet access. If `zarr` isn't preinstalled in the Kaggle
Python image, attach it as a wheel via a Kaggle Dataset input (internet is
disabled at submission time, so `pip install` won't work then).

Tune `DETECTION_PARAMS` / `TRACKING_PARAMS` below against a few training
volumes with known ground truth before relying on this for a leaderboard
submission — thresholds and distance gates are dataset-density dependent.

In [ ]:
import glob
import os
from dataclasses import dataclass

import numpy as np
import pandas as pd
import zarr
from scipy import ndimage as ndi
from scipy.optimize import linear_sum_assignment
from scipy.spatial.distance import cdist
from skimage.feature import peak_local_max
from skimage.filters import threshold_otsu
from skimage.measure import regionprops
from skimage.segmentation import watershed

# Adjust OUTPUT_PATH if needed. TEST_DIR auto-detects a *.zarr folder under
# /kaggle/input further down -- pass it explicitly to run() if that fails.
OUTPUT_PATH = "/kaggle/working/submission.csv"


## Detection

In [ ]:
@dataclass(frozen=True)
class PhysicalScale:
    z: float = 1.625
    y: float = 0.40625
    x: float = 0.40625

    def as_array(self) -> np.ndarray:
        return np.array([self.z, self.y, self.x], dtype=np.float64)


@dataclass
class DetectionParams:
    smooth_sigma_um: float = 1.2
    min_peak_separation_um: float = 4.0
    threshold_rel: float = 0.15
    min_region_voxels: int = 5


def _voxel_sigma(scale: PhysicalScale, sigma_um: float) -> np.ndarray:
    return sigma_um / scale.as_array()


def _voxel_footprint_radius(scale: PhysicalScale, radius_um: float) -> tuple:
    radii = np.maximum(np.round(radius_um / scale.as_array()), 1).astype(int)
    return tuple(radii)


def detect_cells(volume: np.ndarray, scale: PhysicalScale = PhysicalScale(),
                  params: DetectionParams = DetectionParams()) -> np.ndarray:
    """Detect cell centroids in a single (Z, Y, X) volume.

    Returns (N, 4) array of [z, y, x, intensity] with integer voxel centroids,
    or shape (0, 4) if nothing is detected.
    """
    volume = np.asarray(volume, dtype=np.float32)
    if volume.size == 0 or not np.any(volume):
        return np.zeros((0, 4), dtype=np.float64)

    sigma = _voxel_sigma(scale, params.smooth_sigma_um)
    smoothed = ndi.gaussian_filter(volume, sigma=sigma)

    try:
        otsu = threshold_otsu(smoothed[smoothed > 0]) if np.any(smoothed > 0) else 0.0
    except ValueError:
        otsu = 0.0
    dynamic = smoothed.max() * params.threshold_rel
    threshold = max(otsu, dynamic)
    mask = smoothed > threshold
    if not np.any(mask):
        return np.zeros((0, 4), dtype=np.float64)

    footprint_radii = _voxel_footprint_radius(scale, params.min_peak_separation_um)
    footprint = np.ones(tuple(2 * r + 1 for r in footprint_radii), dtype=bool)

    coords = peak_local_max(smoothed, footprint=footprint, labels=mask.astype(np.int32),
                             exclude_border=False)
    if coords.shape[0] == 0:
        return np.zeros((0, 4), dtype=np.float64)

    markers = np.zeros(volume.shape, dtype=np.int32)
    for i, (z, y, x) in enumerate(coords, start=1):
        markers[z, y, x] = i

    labels = watershed(-smoothed, markers=markers, mask=mask)

    detections = []
    for prop in regionprops(labels, intensity_image=volume):
        if prop.area < params.min_region_voxels:
            continue
        centroid = getattr(prop, "centroid_weighted", None)
        if centroid is None:
            centroid = prop.weighted_centroid
        mean_intensity = getattr(prop, "intensity_mean", None)
        if mean_intensity is None:
            mean_intensity = prop.mean_intensity
        z, y, x = centroid
        detections.append((z, y, x, mean_intensity))

    if not detections:
        return np.zeros((0, 4), dtype=np.float64)

    out = np.array(detections, dtype=np.float64)
    out[:, :3] = np.round(out[:, :3])
    return out


## Tracking (linking + division recovery)

In [ ]:
@dataclass
class TrackingParams:
    max_link_dist_um: float = 12.0
    max_division_dist_um: float = 15.0


def _to_physical(points_zyx: np.ndarray, scale: PhysicalScale) -> np.ndarray:
    return points_zyx * scale.as_array()


def link_frames(prev_pts, curr_pts, scale: PhysicalScale, params: TrackingParams):
    """1-1 assignment between prev and curr centroids via Hungarian algorithm.

    Padded with dummy rows/columns (cost = threshold) so cells may appear,
    disappear, or go unmatched instead of being forced into a bad pairing.
    Returns list of (prev_index, curr_index) pairs.
    """
    n, m = len(prev_pts), len(curr_pts)
    if n == 0 or m == 0:
        return []

    prev_um = _to_physical(prev_pts, scale)
    curr_um = _to_physical(curr_pts, scale)
    dist = cdist(prev_um, curr_um)

    threshold = params.max_link_dist_um
    big = threshold * 1000.0 + 1.0

    size = n + m
    cost = np.full((size, size), big, dtype=np.float64)
    cost[:n, :m] = dist
    cost[:n, m:] = np.where(np.eye(n, dtype=bool), threshold, big)
    cost[n:, :m] = np.where(np.eye(m, dtype=bool), threshold, big)
    cost[n:, m:] = 0.0

    row_ind, col_ind = linear_sum_assignment(cost)

    matches = []
    for r, c in zip(row_ind, col_ind):
        if r < n and c < m and dist[r, c] <= threshold:
            matches.append((r, c))
    return matches


def recover_divisions(prev_pts, curr_pts, matches, scale: PhysicalScale, params: TrackingParams):
    """Attach unmatched curr cells to their nearest prev cell (within
    max_division_dist_um), giving an already-matched parent a second
    outgoing edge — i.e. a division."""
    n, m = len(prev_pts), len(curr_pts)
    if n == 0 or m == 0:
        return []

    matched_curr = {c for _, c in matches}
    unmatched_curr = [c for c in range(m) if c not in matched_curr]
    if not unmatched_curr:
        return []

    prev_um = _to_physical(prev_pts, scale)
    curr_um = _to_physical(curr_pts, scale)

    extra = []
    for c in unmatched_curr:
        d = np.linalg.norm(prev_um - curr_um[c], axis=1)
        p = int(np.argmin(d))
        if d[p] <= params.max_division_dist_um:
            extra.append((p, c))
    return extra


@dataclass
class DatasetTracker:
    """Accumulates nodes/edges for one dataset across all timepoints."""
    scale: PhysicalScale
    params: TrackingParams
    nodes: list = None
    edges: list = None
    _next_node_id: int = 1
    _prev_pts: np.ndarray = None
    _prev_node_ids: np.ndarray = None

    def __post_init__(self):
        self.nodes = []
        self.edges = []

    def add_frame(self, t: int, detections: np.ndarray) -> None:
        pts = detections[:, :3] if detections.size else np.zeros((0, 3))
        node_ids = np.arange(self._next_node_id, self._next_node_id + len(pts))
        self._next_node_id += len(pts)

        for nid, (z, y, x) in zip(node_ids, pts):
            self.nodes.append((int(nid), int(t), int(z), int(y), int(x)))

        if self._prev_pts is not None and len(self._prev_pts) and len(pts):
            matches = link_frames(self._prev_pts, pts, self.scale, self.params)
            extra = recover_divisions(self._prev_pts, pts, matches, self.scale, self.params)
            for p_idx, c_idx in matches + extra:
                self.edges.append((int(self._prev_node_ids[p_idx]), int(node_ids[c_idx])))

        self._prev_pts = pts
        self._prev_node_ids = node_ids


## Data access

Handles, in order of preference:

1. **OME-NGFF multiscale groups** (`.zattrs` -> `multiscales`) -- picks the finest resolution level and uses its `axes` metadata (or the t/c-first convention) to index out one `(Z, Y, X)` volume per timepoint.
2. A bare array of shape `(T, Z, Y, X)` or `(T, C, Z, Y, X)` at the root.
3. The legacy convention of one 3D `(Z, Y, X)` array per timepoint, keyed `"0"`, `"1"`, ... directly under a plain group.

Run `inspect_zarr(path)` on a real dataset if a layout doesn't fit any of these and the resulting error doesn't make the actual structure obvious.

In [ ]:
def _zarr_attrs(node) -> dict:
    try:
        return dict(node.attrs)
    except Exception:
        return {}


def _is_zarr_array(node) -> bool:
    return hasattr(node, "shape") and hasattr(node, "dtype")


def _sorted_keys(keys) -> list:
    try:
        return sorted(keys, key=int)
    except ValueError:
        return sorted(keys)


def _axis_names_from_attrs(attrs: dict):
    axes = attrs.get("axes") or attrs.get("_ARRAY_DIMENSIONS")
    if not axes:
        return None
    return [a["name"] if isinstance(a, dict) else str(a) for a in axes]


def _infer_time_channel_axes(ndim: int, axis_names):
    """Return (t_axis, c_axis) for an array of `ndim` dims.

    Uses OME-NGFF-style axis names ("t", "c", "z", "y", "x") when available;
    otherwise falls back to the near-universal convention that time (and
    channel, if present) are the leading axes.
    """
    if axis_names and len(axis_names) == ndim:
        t_axis = axis_names.index("t") if "t" in axis_names else None
        c_axis = axis_names.index("c") if "c" in axis_names else None
        return t_axis, c_axis
    if ndim == 5:
        return 0, 1
    if ndim == 4:
        return 0, None
    if ndim == 3:
        return None, None
    raise ValueError(f"Don\'t know how to interpret a {ndim}-D array without axis metadata")


class ZarrTimeSeries:
    def __init__(self, path: str):
        root = zarr.open(path, mode="r")
        self._mode, self._payload = self._resolve(root)

    @classmethod
    def _resolve(cls, node, axis_names=None):
        if _is_zarr_array(node):
            names = axis_names or _axis_names_from_attrs(_zarr_attrs(node))
            t_axis, c_axis = _infer_time_channel_axes(node.ndim, names)
            return "array", (node, t_axis, c_axis)

        attrs = _zarr_attrs(node)
        multiscales = attrs.get("multiscales")
        if multiscales:
            ms = multiscales[0]
            dataset_path = ms["datasets"][0]["path"]
            axes = ms.get("axes")
            names = [a["name"] if isinstance(a, dict) else str(a) for a in axes] if axes else axis_names
            return cls._resolve(node[dataset_path], names)

        array_keys = _sorted_keys(node.array_keys())
        if array_keys:
            first = node[array_keys[0]]
            if len(array_keys) > 1 and first.ndim == 3:
                return "per_frame_group", (node, array_keys)
            return cls._resolve(first, axis_names)

        group_keys = _sorted_keys(node.group_keys())
        if len(group_keys) == 1:
            return cls._resolve(node[group_keys[0]], axis_names)

        raise ValueError(
            f"Could not find a data array in this zarr (keys={list(node.keys())}). "
            "Run inspect_zarr(path) on it and adjust ZarrTimeSeries._resolve."
        )

    def __len__(self) -> int:
        if self._mode == "array":
            array, t_axis, _ = self._payload
            return array.shape[t_axis] if t_axis is not None else 1
        _, keys = self._payload
        return len(keys)

    def get_frame(self, t: int):
        if self._mode == "per_frame_group":
            node, keys = self._payload
            return node[keys[t]][:]

        array, t_axis, c_axis = self._payload
        if t_axis is None:
            frame = array[:]
        else:
            index = [slice(None)] * array.ndim
            index[t_axis] = t
            frame = array[tuple(index)]

        if c_axis is not None:
            shifted_c_axis = c_axis if t_axis is None or c_axis < t_axis else c_axis - 1
            frame = np.take(frame, 0, axis=shifted_c_axis)

        frame = np.asarray(frame)
        while frame.ndim > 3:
            frame = frame[0]
        return frame


def inspect_zarr(path: str, max_depth: int = 4) -> None:
    """Print a zarr dataset\'s group/array structure, shapes, and attrs."""
    def walk(node, prefix: str, depth: int) -> None:
        if depth > max_depth:
            return
        attrs = _zarr_attrs(node)
        if _is_zarr_array(node):
            print(f"{prefix}[array] shape={node.shape} dtype={node.dtype} attrs={list(attrs.keys())}")
            return
        print(f"{prefix}[group] attrs={list(attrs.keys())}")
        for key in _sorted_keys(node.array_keys()):
            walk(node[key], prefix + f"  {key}/", depth + 1)
        for key in _sorted_keys(node.group_keys()):
            walk(node[key], prefix + f"  {key}/", depth + 1)

    walk(zarr.open(path, mode="r"), prefix="", depth=0)


## Submission formatting

In [ ]:
SUBMISSION_COLUMNS = ["id", "dataset", "row_type", "node_id", "t", "z", "y", "x", "source_id", "target_id"]


def dataset_rows(dataset: str, nodes, edges) -> pd.DataFrame:
    node_rows = [
        {"dataset": dataset, "row_type": "node", "node_id": nid, "t": t, "z": z, "y": y, "x": x,
         "source_id": -1, "target_id": -1}
        for nid, t, z, y, x in nodes
    ]
    edge_rows = [
        {"dataset": dataset, "row_type": "edge", "node_id": -1, "t": -1, "z": -1, "y": -1, "x": -1,
         "source_id": src, "target_id": dst}
        for src, dst in edges
    ]
    return pd.DataFrame(node_rows + edge_rows)


def build_submission(per_dataset_rows) -> pd.DataFrame:
    if not per_dataset_rows:
        return pd.DataFrame(columns=SUBMISSION_COLUMNS)
    df = pd.concat(per_dataset_rows, ignore_index=True)
    df.insert(0, "id", range(len(df)))
    return df[SUBMISSION_COLUMNS]


## Run the pipeline over every test dataset

In [ ]:
SCALE = PhysicalScale(z=1.625, y=0.40625, x=0.40625)
DETECTION_PARAMS = DetectionParams()
TRACKING_PARAMS = TrackingParams()


def track_dataset(zarr_path: str, dataset_name: str) -> pd.DataFrame:
    series = ZarrTimeSeries(zarr_path)
    print(f"[{dataset_name}] {len(series)} timepoint(s) detected "
          f"(mode={series._mode}, shape={getattr(series._payload[0], 'shape', None)})")
    tracker = DatasetTracker(scale=SCALE, params=TRACKING_PARAMS)

    for t in range(len(series)):
        volume = series.get_frame(t)
        detections = detect_cells(volume, SCALE, DETECTION_PARAMS)
        tracker.add_frame(t, detections)

    return dataset_rows(dataset_name, tracker.nodes, tracker.edges)


def discover_test_datasets(test_dir: str):
    paths = sorted(glob.glob(os.path.join(test_dir, "*.zarr")))
    return [(p, os.path.splitext(os.path.basename(p))[0]) for p in paths]


def discover_test_dir(root: str = "/kaggle/input") -> str:
    """Find a folder under `root` containing *.zarr subfolders (the
    competition slug in the mounted path isn\'t known ahead of time)."""
    if not os.path.isdir(root):
        raise FileNotFoundError(f"{root} does not exist")
    candidates = []
    for dirpath, dirnames, _filenames in os.walk(root):
        if any(d.endswith(".zarr") for d in dirnames):
            candidates.append(dirpath)
        dirnames[:] = [d for d in dirnames if not d.endswith(".zarr")]
    if not candidates:
        raise FileNotFoundError(f"No folder containing *.zarr subfolders found under {root}.")
    preferred = [c for c in candidates if os.path.basename(c).lower() == "test"]
    return preferred[0] if preferred else candidates[0]


TEST_DIR = discover_test_dir()
print(f"Auto-detected test_dir: {TEST_DIR}")

datasets = discover_test_datasets(TEST_DIR)
assert datasets, f"No .zarr datasets found under {TEST_DIR}"

per_dataset = []
for zarr_path, name in datasets:
    print(f"[{name}] tracking...")
    rows = track_dataset(zarr_path, name)
    n_nodes = (rows["row_type"] == "node").sum()
    n_edges = (rows["row_type"] == "edge").sum()
    print(f"[{name}] {n_nodes} nodes, {n_edges} edges")
    per_dataset.append(rows)

submission = build_submission(per_dataset)
submission.to_csv(OUTPUT_PATH, index=False)
print(f"Wrote {OUTPUT_PATH} ({len(submission)} rows)")
submission.head()


## Notes for improving your score

- **Tune on train data first.** `DetectionParams.threshold_rel` /
  `min_peak_separation_um` and `TrackingParams.max_link_dist_um` /
  `max_division_dist_um` are density- and noise-dependent; sweep them against
  a few annotated training volumes and pick values that maximize the edge +
  division Jaccard before trusting this on test data.
- **Swap in a learned detector.** The classical watershed detector is a
  dependency-free baseline. A 3D StarDist / U-Net segmentation model
  (trained offline, weights loaded from a Kaggle Dataset input since
  internet is disabled at submission time) will typically detect dense,
  irregularly-shaped cells far more reliably.
- **Smarter linking.** Consider incorporating appearance/intensity features
  into the assignment cost, or a short temporal window (e.g. skip-frame
  linking) to bridge missed detections, rather than pure nearest-neighbor
  distance on two consecutive frames.
- **Division precision.** `recover_divisions` is a simple nearest-parent
  heuristic; false divisions from detection noise will hurt the division
  Jaccard. Consider requiring the parent to be within a stricter radius, or
  validating candidate divisions against expected daughter symmetry/size.